In [2]:
import os
import sys
import glob
import pandas as pd

#avoid import errors
parent_dir=os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(parent_dir)

from scripts.CDS_machine import CDS_2_gffcomp
import scripts.get_busco_db as bob
import scripts.get_genetic_code as ggc
from scripts.counting_machine import write_counts, CATEGORIES


In [3]:
raw_data="../../data/species"
query_email="wsxktjnfwcecxnotrf@gonrr.net"
busco_database="/no_backup/rg/references/busco_downloads"

!ls $raw_data

Acanthamoeba_polyphaga_5757		   Micromonas_pusilla_CCMP1545_38833
Babesia_duncani_323732			   Naegleria_gruberi_5762
Caulerpa_lentillifera_148947		   Paramecium_bursaria_74790
Chaetoceros_neogracilis_240364		   Paramecium_tetraurelia_5888
Chlamydomonas_reinhardtii_3055		   Phytophthora_agathidicida_1642459
Chlorella_sorokiniana_3076		   Plasmodium_berghei_ANKA_5821
Cladocopium_goreaui_2562237		   Plasmodium_falciparum_3D7_36329
Conticribra_weissflogii_1577725		   Plasmodium_ovale_36330
Cryptosporidium_parvum_Iowa_II_5807	   Plasmodium_vivax_5855
Cyanidiococcus_yangmingshanensis_2690220   Plasmodium_yoelii_yoelii_73239
Cyanidioschyzon_merolae_strain_10D_280699  Pseudo-nitzschia_delicatissima_44447
Cyanidium_caldarium_2771		   Pseudo-nitzschia_multiseries_37319
Cylindrotheca_closterium_2856		   Pseudo-nitzschia_pungens_37318
Dunaliella_salina_3046			   Pyrocystis_lunula_2972
Eimeria_necatrix_51315			   Pyropia_haitanensis_1262161
Entamoeba_histolytica_HM-1:IMSS_5759	   Pyropia_yezoensis_

# Run GeneID

In [4]:
#geneid_command="time geneid -3P {parameter_file} {assembly} > {annotation.gff3}"

#units = <species>_<source>, taken from the trained parameter files
trained_units=[f.replace(".geneid.optimized.param", "") for f in os.listdir("../results/trainedParams") if f.endswith(".geneid.optimized.param")]
print(len(trained_units))

#checker(if things in file)
fileWritten=False
filepath="../job_commands/total_preds.txt"
with open(filepath, "w") as out:
    #create predictions folder command
    pred_folder="../results/pred"
    print(f"mkdir -p {pred_folder}", file=out)
    for unit in trained_units:
        sp=unit.rsplit("_", 1)[0]

        #shared cleaned genome for the species; model named by unit (source-tagged)
        ref=!realpath ../training_data/species/$sp/CLEAN_*.fna
        model=f"{pred_folder}/{unit}.gff3"

        #own-trained parameters for this unit
        param=os.path.abspath(f"../results/trainedParams/{unit}.geneid.optimized.param")
        fileWritten=True

        #geneid param fasta_assembly > gff_model
        command=f"time geneid -3P {param} {ref[0]} > {model}"

        #link inside the unit folder just in case
        logsDir_cmd=f"mkdir -p ../results/specie_logs/{unit}/" #if running from exsting parameters create logs folder
        lnPred_cmd=f"ln -vf {model} ../results/specie_logs/{unit}/"

        print(command)
        print(command, file=out)
        print(logsDir_cmd, file=out)
        print(lnPred_cmd, file=out)
        #in the end of the execution delete empty files
        print(f"find {pred_folder} -size 0 -print -delete", file=out)

if not fileWritten: #if the file is not written, delete it
    os.remove(filepath)
else: #copy the matching array submitter next to its command file
    !cp ../scripts/memory/predictor.sh ../job_commands/

#Execute commands to predict CDS

70
time geneid -3P /users/rg/jizquierdo/git/geneid-training/results/trainedParams/Paramecium_tetraurelia_5888_reference.geneid.optimized.param /users/rg/jizquierdo/git/geneid-training/training_data/species/Paramecium_tetraurelia_5888/CLEAN_GCF_000165425.1_ASM16542v1_genomic.fna > ../results/pred/Paramecium_tetraurelia_5888_reference.gff3
time geneid -3P /users/rg/jizquierdo/git/geneid-training/results/trainedParams/Giardia_muris_5742_lyric.geneid.optimized.param /users/rg/jizquierdo/git/geneid-training/training_data/species/Giardia_muris_5742/CLEAN_GCA_006247105.1_UU_GM_1.1_genomic.fna > ../results/pred/Giardia_muris_5742_lyric.gff3
time geneid -3P /users/rg/jizquierdo/git/geneid-training/results/trainedParams/Dunaliella_salina_3046_reference.geneid.optimized.param /users/rg/jizquierdo/git/geneid-training/training_data/species/Dunaliella_salina_3046/CLEAN_GCA_002284615.2_Dunsal1_v._2_genomic.fna > ../results/pred/Dunaliella_salina_3046_reference.gff3
time geneid -3P /users/rg/jizquierd

In [5]:
!ls $pred_folder

Babesia_duncani_323732_lyric.gff3
Babesia_duncani_323732_reference.gff3
Chaetoceros_neogracilis_240364_lyric.gff3
Chaetoceros_neogracilis_240364_reference.gff3
Chlamydomonas_reinhardtii_3055_isoquant.gff3
Chlamydomonas_reinhardtii_3055_lyric.gff3
Chlamydomonas_reinhardtii_3055_reference.gff3
Chlorella_sorokiniana_3076_lyric.gff3
Conticribra_weissflogii_1577725_lyric.gff3
Conticribra_weissflogii_1577725_reference.gff3
Cryptosporidium_parvum_Iowa_II_353152_reference.gff3
Cryptosporidium_parvum_Iowa_II_5807_lyric.gff3
Cyanidiococcus_yangmingshanensis_2690220_lyric.gff3
Cyanidiococcus_yangmingshanensis_2690220_reference.gff3
Cyanidioschyzon_merolae_strain_10D_280699_lyric.gff3
Cyanidioschyzon_merolae_strain_10D_280699_reference.gff3
Cyanidium_caldarium_2771_lyric.gff3
Cyanidium_caldarium_2771_reference.gff3
Cylindrotheca_closterium_2856_lyric.gff3
Cylindrotheca_closterium_2856_reference.gff3
Dunaliella_salina_3046_lyric.gff3
Dunaliella_salina_3046_reference.gff3
Eimeria_necatrix_51315_lyri

# Prediction analysis

## BUSCO assessment

In [6]:
#agat for gff>prot
#evaluate every prediction file: <species>_<source>.gff3
pred_units=[os.path.basename(p)[:-len(".gff3")] for p in sorted(glob.glob("../results/pred/*.gff3"))]
print(len(pred_units))

#checker(if things in file)
fileWritten=False
filepath="../job_commands/buscoScoring.txt"
with open(filepath, "w") as f:
    print('cpus="${SLURM_CPUS_PER_TASK:-$(nproc)}"', file=f)
    print(f"mkdir -p ../results/summary/regular/busco_lineage ../results/summary/regular/busco_eukaryote", file=f)
    for unit in pred_units:
        try:
            sp=unit.rsplit("_", 1)[0]
            pred=f"../results/pred/{unit}.gff3"

            fileWritten=True

            #reference sequence (shared genome for the species)
            RefSeq=!realpath ../training_data/species/$sp/CLEAN_*.fna
            RefSeq=RefSeq[0]

            #create folder to store outputs
            prot_res=f"../results/specie_logs/{unit}/{unit}_prot.fa" #protein sequence output
            busco_outPath=f"../results/busco/" #busco comparisons output
            Lbusco_res=f"{unit}_Lbusco" #taxon-lineage busco out name
            Ebusco_res=f"{unit}_Ebusco" #eukaryota busco out name

            #get species taxon
            taxID=sp[sp.rfind("_")+1:]
            #find lineage
            busco_lineage=bob.taxon2lineage(taxID, query_email, f"{busco_database}/file_versions.tsv", "odb12")
            euk_lineage="eukaryota_odb12"
            #resolve the NCBI nuclear genetic code for this taxon (table 1 if unresolved)
            gcode=ggc.taxon2geneticcode(taxID, query_email) or 1

            #gff to protein command
            #per-task AGAT config so parallel array tasks don't collide on agat_config.yaml
            agat_cfg=f"agat_{unit}.yaml"
            expose_cmd=f"agat config --expose --output {agat_cfg} >/dev/null 2>&1"
            print(expose_cmd); print(expose_cmd, file=f)
            TOprotein_cmd=f"agat_sp_extract_sequences.pl -g {pred} -f {RefSeq} -t cds -p --table {gcode} --config {agat_cfg} -o {prot_res}"

            print(TOprotein_cmd)
            print(TOprotein_cmd, file=f)
            clean_cmd=f"rm -f {agat_cfg}"
            print(clean_cmd); print(clean_cmd, file=f)
            #deduplicate protein IDs (AGAT can emit duplicate names that break BUSCO)
            ND_prot_res=prot_res.replace(".fa", "_ND.fa")
            rename_cmd=f"seqkit rename -n {prot_res} > {ND_prot_res}"
            print(rename_cmd)
            print(rename_cmd, file=f)
            
            Lbusco_cmd=f"busco -m protein \
                        -i {ND_prot_res} \
                        --download_path {busco_database} \
                        -l {busco_lineage} \
                        --cpu $cpus \
                        -f \
                        --opt-out-run-stats \
                        --out_path {busco_outPath} \
                        -o {Lbusco_res}" #--offline #autolineage creates errors
            Lbusco_cmd=Lbusco_cmd.replace("                             ", " ")

            Ebusco_cmd=f"busco -m protein \
                        -i {ND_prot_res} \
                        --download_path {busco_database} \
                        -l {euk_lineage} \
                        --cpu $cpus \
                        -f \
                        --opt-out-run-stats \
                        --out_path {busco_outPath} \
                        -o {Ebusco_res}" #--offline #autolineage creates errors
            Ebusco_cmd=Ebusco_cmd.replace("                             ", " ")

            print(Lbusco_cmd); print(Lbusco_cmd, file=f)
            print(Ebusco_cmd); print(Ebusco_cmd, file=f)

            #collect each run JSON into its own summary folder (isoquant naming)
            print(f"ln -vf {busco_outPath}{Lbusco_res}/short_summary.specific.*.json ../results/summary/regular/busco_lineage/{Lbusco_res}.json", file=f)
            print(f"ln -vf {busco_outPath}{Ebusco_res}/short_summary.specific.*.json ../results/summary/regular/busco_eukaryote/{Ebusco_res}.json", file=f)

        except Exception as e:
            print(f"{unit} presents error: {e}")
            continue
        
if not fileWritten: #if the file is not written, delete it
    os.remove(filepath)
else: #copy the matching array submitter next to its command file
    !cp ../scripts/memory/busco.sh ../job_commands/
            

70
agat config --expose --output agat_Babesia_duncani_323732_lyric.yaml >/dev/null 2>&1
agat_sp_extract_sequences.pl -g ../results/pred/Babesia_duncani_323732_lyric.gff3 -f /users/rg/jizquierdo/git/geneid-training/training_data/species/Babesia_duncani_323732/CLEAN_GCF_028658345.1_Bduncani_v1_genomic.fna -t cds -p --table 1 --config agat_Babesia_duncani_323732_lyric.yaml -o ../results/specie_logs/Babesia_duncani_323732_lyric/Babesia_duncani_323732_lyric_prot.fa
rm -f agat_Babesia_duncani_323732_lyric.yaml
seqkit rename -n ../results/specie_logs/Babesia_duncani_323732_lyric/Babesia_duncani_323732_lyric_prot.fa > ../results/specie_logs/Babesia_duncani_323732_lyric/Babesia_duncani_323732_lyric_prot_ND.fa
busco -m protein                         -i ../results/specie_logs/Babesia_duncani_323732_lyric/Babesia_duncani_323732_lyric_prot_ND.fa                         --download_path /no_backup/rg/references/busco_downloads                         -l piroplasmida_odb12                         --c

# Summary

The **regular** evaluation outputs are collected under `results/summary/regular/`: the busco cells above write the BUSCO summaries

In [7]:
#metrics for the regular predictions -> results/summary/regular/counts/ per-species file + counts_summary.tsv.

results_dir = "../results"
summary_dir = f"{results_dir}/summary"
os.makedirs(summary_dir, exist_ok=True)

write_counts(results_dir, summary_dir, "regular", CATEGORIES["regular"])

[regular] metrics: 49 species (70 units) -> counts/ (+ counts_summary.tsv)


([['Babesia_duncani_323732',
   '3457',
   '3457',
   '10408054',
   'NA',
   'NA',
   '332.15',
   '332.15',
   '1.000',
   'NA'],
  ['Chaetoceros_neogracilis_240364',
   '13886',
   '13886',
   '38617829',
   'NA',
   'NA',
   '359.57',
   '359.57',
   '1.000',
   'NA'],
  ['Chlamydomonas_reinhardtii_3055',
   '22322',
   '22322',
   '111098438',
   'NA',
   'NA',
   '200.92',
   '200.92',
   '1.000',
   'NA'],
  ['Chlorella_sorokiniana_3076',
   '12446',
   '12446',
   '38807484',
   'NA',
   'NA',
   '320.71',
   '320.71',
   '1.000',
   'NA'],
  ['Conticribra_weissflogii_1577725',
   '47212',
   '47212',
   '129999045',
   'NA',
   'NA',
   '363.17',
   '363.17',
   '1.000',
   'NA'],
  ['Cryptosporidium_parvum_Iowa_II_353152',
   '3109',
   '3109',
   '9102324',
   'NA',
   'NA',
   '341.56',
   '341.56',
   '1.000',
   'NA'],
  ['Cryptosporidium_parvum_Iowa_II_5807',
   '3243',
   '3243',
   '9102324',
   'NA',
   'NA',
   '356.28',
   '356.28',
   '1.000',
   'NA'],
  ['Cyanidi